<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Before testing any signal, this section looks at the raw distributions of key fields to check for heavy tails that could distort a simple average-based comparison.

impressions_90d shows the most extreme tail: the 90th percentile is 12,136, but the 99th percentile jumps to 73,506 — and the maximum reaches 517,715, roughly 7x the 99th percentile. A small number of very high-traffic pages could dominate any mean-based summary of this field, so median or percentile-based comparisons are safer than raw averages when interpreting impressions_90d.

ctr and engagement_rate show a similar pattern at smaller scale — both have a 99th percentile roughly 10x their 90th percentile, and both cap at 100 (the maximum possible percentage), meaning a handful of pages sit at the theoretical ceiling. avg_position is moderately tailed but less extreme (p99 is roughly double p90). word_count is the most well-behaved field here — its p90-to-p99 ratio is modest (5,327 to 7,292), and it has no missing values in the same distorting way once non-null (25.7% of rows lack it entirely, as found in the leakage-check notebook, but where present, the distribution is reasonably even).

Practical implication: any signal test in this notebook that compares average impressions, CTR, or engagement rate across groups should be read with this skew in mind — a few extreme pages can make a group's average look stronger or weaker than what a "typical" page in that group experiences.

In [8]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git
%cd flyrank-ml-internship

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = (df["trend_direction"] == "down").astype(int)

key_fields = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "word_count"]

print(df[key_fields].describe())

print("\n90th vs 99th percentile (checking for heavy tails):")
for col in key_fields:
    p90 = df[col].quantile(0.90)
    p99 = df[col].quantile(0.99)
    p_max = df[col].max()
    tail_ratio = p99 / p90 if p90 > 0 else float("nan")
    print(f"  {col}: p90={p90:.2f}, p99={p99:.2f}, max={p_max:.2f}, p99/p90 ratio={tail_ratio:.1f}x")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 201, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 201 (delta 87), reused 97 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (201/201), 1.97 MiB | 5.69 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
       impressions_90d           ctr  avg_position  engagement_rate  \
count     30000.000000  30000.000000   30000.00000     30000.000000   
mean       5200.366300      0.510733      16.34238         2.534520   
std       16838.019547      3.279162      15.21679         8.310096   
min           1.000000      0.000000       0.00000         0.000000   
25%          81.000000      0.000000       6.20000         0.000000   
50%         731.000000      0.070000      10.80000         0.000000   
75%        3615.250000      0.290000      22.30000         1.350000   
m

## 2. Signal test #1 / #2 / #3 (verdict each)


**Signal 1** — Word count vs. decline rate: OPPOSITE. Decline rate rises steadily from 44.9% (under 1,500 words) to 59.7% (3,500+ words), a consistent, monotonic pattern backed by solid sample sizes in every bucket (smallest n=2,946). This runs counter to a common assumption that longer content is inherently more durable — in this dataset, longer pages actually decline more often, not less.

**Signal 2** — Engagement rate vs. decline rate: FALSE. Decline rate stays within a narrow band (52.5%-56.5%) across all engagement buckets, showing no meaningful separation despite reasonable sample sizes. Engagement rate alone does not appear to predict decline in this dataset.

**Signal 3** — AI traffic presence vs. decline rate: FALSE. Pages with any AI-referral traffic decline at 55.3%, barely different from the 54.1% rate for pages with none. Only 1,930 of 30,000 pages (6.4%) have any AI traffic at all, a small subgroup that further limits how much weight this comparison can carry.

In [9]:
df["word_count_bucket"] = pd.cut(df["word_count"], bins=[0, 1500, 2500, 3500, 10000],
                                   labels=["<1500", "1500-2500", "2500-3500", "3500+"])
signal1 = df.groupby("word_count_bucket", observed=True).agg(
    n=("content_id", "count"), pct_declining=("needs_review", lambda x: x.mean() * 100)
).round(1)
print("SIGNAL 1 — Word Count vs. Decline Rate — Verdict: OPPOSITE")
print(signal1)

df["engagement_bucket"] = pd.cut(df["engagement_rate"], bins=[-0.01, 0, 1, 5, 100],
                                   labels=["0 (none)", "0-1", "1-5", "5+"])
signal2 = df.groupby("engagement_bucket", observed=True).agg(
    n=("content_id", "count"), pct_declining=("needs_review", lambda x: x.mean() * 100)
).round(1)
print("\nSIGNAL 2 — Engagement Rate vs. Decline Rate — Verdict: FALSE")
print(signal2)

df["has_ai_traffic"] = df["ai_sessions_90d"] > 0
signal3 = df.groupby("has_ai_traffic").agg(
    n=("content_id", "count"), pct_declining=("needs_review", lambda x: x.mean() * 100)
).round(1)
print("\nSIGNAL 3 — AI Traffic Presence vs. Decline Rate — Verdict: FALSE")
print(signal3)

SIGNAL 1 — Word Count vs. Decline Rate — Verdict: OPPOSITE
                      n  pct_declining
word_count_bucket                     
<1500              3228           44.9
1500-2500          2946           57.4
2500-3500          9842           58.8
3500+              6285           59.7

SIGNAL 2 — Engagement Rate vs. Decline Rate — Verdict: FALSE
                       n  pct_declining
engagement_bucket                      
0 (none)           21629           54.4
0-1                  519           56.5
1-5                 3861           54.4
5+                  3991           52.5

SIGNAL 3 — AI Traffic Presence vs. Decline Rate — Verdict: FALSE
                    n  pct_declining
has_ai_traffic                      
False           28070           54.1
True             1930           55.3


## 3. The flag-linked test

FlyRank's refresh flag rests on the assumption that older, un-updated content declines more often, the longer a page goes without a refresh, the more likely it is to be losing performance. This test re-examines that assumption using the same freshness-tier bucketing approach as the Week 4 baseline notebook, as an independent check.

The pattern replicates exactly: decline rate rises from 51.1% (0-30 days) to 61.1% (91-180 days) consistent with the refresh flag's core assumption but then reverses to 47.1% at 181+ days, the lowest rate of any tier, on a small sample (n=174).

Verdict: MIXED. The refresh flag's assumption holds for content up to roughly six months old, where the highest-confidence risk window is 91-180 days (n=9,171, a robust sample). But the assumption breaks down for the oldest content: pages surviving past 181 days without an update do not show elevated decline risk in this data; if anything, the small sample suggests the opposite. This reinforces the Week 4 finding that FlyRank's refresh flag would be more precise if its staleness window were narrowed to 91-180 days specifically, rather than applied as "older is always riskier."

In [10]:
staleness_test = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    pct_declining=("needs_review", lambda x: x.mean() * 100)
).round(1)

print("FLAG-LINKED TEST — Staleness (refresh flag assumption)")
print("Assumption: older, un-updated content declines more often.")
print(staleness_test)
print("\nVerdict: MIXED — assumption holds through 91-180 days (n=9,171),")
print("but reverses at 181+ days (n=174, small sample).")

FLAG-LINKED TEST — Staleness (refresh flag assumption)
Assumption: older, un-updated content declines more often.
                    n  pct_declining
freshness_tier                      
0-30            20480           51.1
181+              174           47.1
31-90             175           58.9
91-180           9171           61.1

Verdict: MIXED — assumption holds through 91-180 days (n=9,171),
but reverses at 181+ days (n=174, small sample).


## 4. What this means in practice


A content review team should treat the 91-180 day freshness window as the highest-confidence staleness signal for prioritizing refresh candidates,not "older content is always riskier," since that assumption breaks down past 181 days in this data. Word count should be read with caution: longer pages decline more often here, the opposite of what might be assumed, so depth alone should not be treated as a protective signal without further investigation into why. Engagement rate and AI traffic presence, on their own, showed no meaningful relationship with decline in this dataset and should not be used as standalone flags, they may still be useful combined with other signals, but neither carried a clear pattern in isolation.

In [11]:
verdict_summary = pd.DataFrame([
    {"signal": "Word count vs. decline", "verdict": "OPPOSITE", "note": "Longer content declines more, not less"},
    {"signal": "Engagement rate vs. decline", "verdict": "FALSE", "note": "No meaningful separation across buckets"},
    {"signal": "AI traffic presence vs. decline", "verdict": "FALSE", "note": "Minimal difference; small subgroup (6.4%)"},
    {"signal": "Staleness vs. decline (flag-linked)", "verdict": "MIXED", "note": "Holds through 91-180 days, reverses at 181+"}
])

print("SIGNAL AUDIT SUMMARY")
print(verdict_summary.to_string(index=False))

SIGNAL AUDIT SUMMARY
                             signal  verdict                                        note
             Word count vs. decline OPPOSITE      Longer content declines more, not less
        Engagement rate vs. decline    FALSE     No meaningful separation across buckets
    AI traffic presence vs. decline    FALSE   Minimal difference; small subgroup (6.4%)
Staleness vs. decline (flag-linked)    MIXED Holds through 91-180 days, reverses at 181+


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.